# 第3章：基于直方图统计的处理

## 编程实践：手写直方图统计 / 均衡化 / 匹配

| 项目 | 说明 |
|------|------|
| 输入图片 | `lenaface.jpg`（灰度）；`test1.png` + `test2.png`（直方图匹配源/参考） |
| 手写核心 | 直方图统计、CDF、均衡化、直方图匹配 |
| 允许调用 | 仅 `cv_imread` / `cv_imwrite` 图像读写 |
| 对比验证 | 与 OpenCV `cv2.equalizeHist` 结果做数值误差对比 |


## 一、学习目标

1. 掌握如何提取一张图像的**像素直方图**。
2. 掌握**直方图均衡化**的实现方式与效果特点。
3. 掌握**直方图匹配**（源图 + 参考图）的实现方式与效果特点。


## 二、原理与公式

### 直方图

对灰度图像统计每个灰度级出现的次数：

```
hist[k] = count(I[y, x] == k),  k = 0..255
```

直方图是对整幅图的**全局统计**，丢失空间位置信息，但能反映对比度与亮度分布。

### 直方图均衡化

先求累积分布函数 CDF，再把 CDF 映射到 `[0,255]`：

```
cdf[k] = sum(hist[0..k]) / total_pixels
lut[k] = round(cdf[k] * 255)
out[y, x] = lut[in[y, x]]
```

均衡化使 CDF 尽量接近直线，从而拉伸对比度；对偏暗/偏亮、对比度低的图像效果明显。

### 直方图匹配

让源图的直方图形状逼近参考图。对每个灰度级 `i`，在参考 CDF 中找最接近的 `j`：

```
lut[i] = argmin_j | ref_cdf[j] - src_cdf[i]|
out[y, x] = lut[in[y, x]]
```


## 三、手写约束清单

- ✅ 允许：`cv_imread` / `cv_imwrite`；Python 循环与算术；`np.zeros` 开辟空间。
- ❌ 禁止：`np.histogram` / `cv2.calcHist` / `cv2.equalizeHist` 用于实现（这些仅可用于对比验证）。
- ✅ 可视化：`matplotlib` 仅用于显示直方图与图像。


In [ ]:
import sys
from pathlib import Path

# 向上查找项目根目录（含 utils.py），并加入 sys.path
ROOT = Path.cwd().resolve()
while not (ROOT / "utils.py").exists():
    if ROOT.parent == ROOT:
        raise FileNotFoundError("未找到项目根目录 utils.py")
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import numpy as np
import cv2
import matplotlib.pyplot as plt

from utils import cv_imread, cv_imwrite, set_random_seed, setup_plot_chinese, show_images, compare_results

setup_plot_chinese()
set_random_seed(42)
print(f"OpenCV 版本: {cv2.__version__}")
print(f"当前工作目录: {Path.cwd()}")


In [ ]:
def to_grayscale_manual(image):
    """手写灰度化：Y = 0.299R + 0.587G + 0.114B。"""
    h, w, _ = image.shape
    gray = np.zeros((h, w), dtype=np.uint8)
    for y in range(h):
        for x in range(w):
            b, g, r = (float(v) for v in image[y, x, :])
            gray[y, x] = int(round(0.299 * r + 0.587 * g + 0.114 * b))
    return gray


def compute_hist_manual(gray):
    """手写直方图统计，返回长度 256 的频数数组。"""
    hist = np.zeros(256, dtype=np.int64)
    for y in range(gray.shape[0]):
        for x in range(gray.shape[1]):
            hist[int(gray[y, x])] += 1
    return hist


def compute_cdf(hist, total):
    """由直方图计算归一化累积分布函数 CDF。"""
    cdf = np.zeros(256, dtype=np.float64)
    acc = 0
    for i in range(256):
        acc += hist[i]
        cdf[i] = acc / total
    return cdf


def histogram_equalization_manual(gray):
    """手写直方图均衡化。"""
    hist = compute_hist_manual(gray)
    total = gray.shape[0] * gray.shape[1]
    cdf = compute_cdf(hist, total)

    lut = np.zeros(256, dtype=np.uint8)
    for i in range(256):
        lut[i] = int(round(cdf[i] * 255.0))

    h, w = gray.shape
    out = np.zeros((h, w), dtype=np.uint8)
    for y in range(h):
        for x in range(w):
            out[y, x] = lut[int(gray[y, x])]
    return out, hist, lut


def histogram_matching_manual(source, reference):
    """手写直方图匹配：把 source 的直方图逼近 reference 的直方图。"""
    src_hist = compute_hist_manual(source)
    ref_hist = compute_hist_manual(reference)
    src_cdf = compute_cdf(src_hist, source.shape[0] * source.shape[1])
    ref_cdf = compute_cdf(ref_hist, reference.shape[0] * reference.shape[1])

    lut = np.zeros(256, dtype=np.uint8)
    for i in range(256):
        best_j, best_dist = 0, float("inf")
        for j in range(256):
            dist = abs(ref_cdf[j] - src_cdf[i])
            if dist < best_dist:
                best_dist = dist
                best_j = j
        lut[i] = best_j

    h, w = source.shape
    out = np.zeros((h, w), dtype=np.uint8)
    for y in range(h):
        for x in range(w):
            out[y, x] = lut[int(source[y, x])]
    return out, src_hist, ref_hist


In [ ]:
# 读取灰度图，手写直方图统计与均衡化
color = cv_imread("lenaface.jpg", cv2.IMREAD_COLOR)
assert color is not None, "读取 lenaface.jpg 失败"
gray = to_grayscale_manual(color)

eq, hist, lut = histogram_equalization_manual(gray)

# ---------- 与 OpenCV 对比验证 ----------
eq_cv = cv2.equalizeHist(gray)
compare_results(eq, eq_cv, "直方图均衡化")
cv_imwrite("equalized_image.jpg", eq)

# 可视化直方图与结果
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].bar(range(256), hist, width=1.0, color="steelblue")
axes[0].set_title("原图直方图")
axes[1].bar(range(256), compute_hist_manual(eq), width=1.0, color="darkorange")
axes[1].set_title("均衡化后直方图")
plt.tight_layout()
plt.show()

show_images([gray, eq], ["原灰度图", "直方图均衡化"], figsize=(9, 4))


In [ ]:
# 手写直方图匹配：把 test1.png 的灰度分布逼近 test2.png
src_color = cv_imread("test1.png", cv2.IMREAD_COLOR)
ref_color = cv_imread("test2.png", cv2.IMREAD_COLOR)
src = to_grayscale_manual(src_color)
ref = to_grayscale_manual(ref_color)

matched, src_hist, ref_hist = histogram_matching_manual(src, ref)
cv_imwrite("histogram_matched.jpg", matched)

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
axes[0].bar(range(256), src_hist, width=1.0, color="steelblue")
axes[0].set_title("源图直方图")
axes[1].bar(range(256), ref_hist, width=1.0, color="darkorange")
axes[1].set_title("参考图直方图")
axes[2].bar(range(256), compute_hist_manual(matched), width=1.0, color="seagreen")
axes[2].set_title("匹配后直方图")
plt.tight_layout()
plt.show()

show_images([src, ref, matched], ["源图", "参考图", "直方图匹配结果"], figsize=(13, 4))


## 四、结果与参数分析

- 均衡化把 CDF 线性化，整幅图对比度增强；直方图会从聚集变为**更分散**。
- 匹配让源图灰度分布逼近参考图，可理解为"按参考图的色调风格重映射源图"。
- 直方图方法只改变灰度映射（**像素值重映射**），不改变像素位置，因此不引入几何失真。

**易错点**
1. CDF 必须用**总像素数**归一化，否则 LUT 溢出。
2. 匹配时的最近邻搜索要处理 CDF 相等的情况，优先取较小的 `j`。
3. 彩色图应先分离通道或转灰度再处理，避免把三通道混在一起统计。


## 五、科研规范小结

1. **统计量与算法分离**：`compute_hist_manual` / `compute_cdf` 各自单一职责。
2. **可复现**：直方图方法本身无随机性，但统一使用 `set_random_seed` 便于与其它章节保持一致。
3. **对比验证**：与 `cv2.equalizeHist` 做数值对比，误差应为 0 或仅因取整产生 1 个灰度级。


## 六、练习：彩色图三通道分别均衡化

**要求**：对 `lena.jpeg` 的 B、G、R 三通道分别做直方图均衡化，再合并显示；观察与"转灰度后均衡化"的差异。


In [ ]:
# ==================== 练习解决方案 ====================
def equalize_color_manual(image):
    """对 BGR 三通道分别做直方图均衡化。"""
    h, w, _ = image.shape
    out = np.zeros_like(image)
    for ch in range(3):
        out[:, :, ch] = histogram_equalization_manual(image[:, :, ch])[0]
    return out

img = cv_imread("lena.jpeg", cv2.IMREAD_COLOR)
out_color = equalize_color_manual(img)
show_images([img, out_color], ["原图", "三通道分别均衡化"], figsize=(9, 4))
